# Banking Customer Behaviour & Asset Analytics
### Extraction → profiling → justified cleaning → segmentation → interactive analysis

**Deliverables:** this reproducible cleaning notebook, exported datasets/audit reports, and the BankScope dashboard website.

The default source is `data/banking_raw.csv`, copied from the supplied 70-country file. Its `DataOrigin` explicitly says **Synthetic classroom simulation**. It is not live banking data. The alternative `data/bank_churn_original.csv` is supplied separately and is not merged with it.

**Important:** imputation cannot recover the true missing value. It is a documented assumption for exploratory analysis. Never impute customer IDs, outcomes, exchange rates or provenance. Preserve the raw file and report unresolved issues instead of forcing completeness to 100%.

Run all cells in order. Python dependencies: pandas, numpy, plotly, openpyxl (see `requirements.txt`). The utilities below are embedded so this notebook can travel with just its input CSV.

In [ ]:
from pathlib import Path
import hashlib, json, sys
import pandas as pd
import numpy as np
import plotly.express as px
from IPython.display import display, Markdown
pd.set_option('display.max_columns', 40)
ROOT = Path.cwd()
DATA_PATH = ROOT / 'data' / 'banking_raw.csv'
if not DATA_PATH.exists():
    candidates = [ROOT/'banking_raw.csv', ROOT/'banking_70_countries_10000_messy.csv']
    DATA_PATH = next((p for p in candidates if p.exists()), DATA_PATH)
assert DATA_PATH.exists(), 'Place the supplied CSV in data/banking_raw.csv, or change DATA_PATH.'
OUT = ROOT / 'outputs'
OUT.mkdir(exist_ok=True)
print('Input:', DATA_PATH.name)
print('SHA-256:', hashlib.sha256(DATA_PATH.read_bytes()).hexdigest())
print('Python:', sys.version.split()[0], '| pandas:', pd.__version__, '| numpy:', np.__version__)

## 1. Reusable, auditable pipeline utilities
This cell embeds the same tested engine used by the website. It defines extraction, schema mapping, missing-token recognition, domain validation, cleaning, transformations, filtering, segmentation and safe exports.

**Project validation assumptions:** adult age 18–110, credit score 300–850, tenure 0–80 and no greater than age, integer-valued counts, nonnegative recorded holdings, binary fields 0/1, positive FX and Bitcoin price. These are configurable project rules, not universal banking laws. Invalid cells become missing in an analytical copy; original strings remain available for review.

Empty/whitespace strings, NA, N/A, null, NaN, ?, -, missing and not available are missing markers. `None` means no loan when it appears in loan type; it is not discarded there. Valid zero balances remain zero.

In [ ]:
"""Shared, deterministic analytics pipeline for the notebook and local website."""
import io
import re
from datetime import datetime, timezone
import numpy as np
import pandas as pd

MISSING = {'', 'na', 'n/a', 'null', 'nan', '?', '-', 'missing', 'not available'}
ALIASES = {'customerid':'customer_id', 'geography':'country', 'creditscore':'credit_score',
 'numofproducts':'products_number', 'numofproduct':'products_number', 'numproducts':'products_number',
 'hascreditcard':'credit_card', 'isactivemember':'active_member', 'estimatedsalary':'estimated_salary',
 'exited':'churn', 'fxtousd':'fx_to_usd', 'bitcoinholdingsbtc':'bitcoin_btc',
 'bitcoinpriceusd':'bitcoin_price_usd', 'fixeddepositbalance':'fixed_deposit',
 'fixeddepositratepct':'fd_rate_pct', 'fixeddeposittermmonths':'fd_term_months',
 'loanbalance':'loan_balance', 'loantype':'loan_type', 'loanratepct':'loan_rate_pct',
 'stocksbalance':'stocks', 'mutualfundsbalance':'mutual_funds', 'hasbitcoin':'has_bitcoin',
 'hasfixeddeposit':'has_fixed_deposit', 'hasloan':'has_loan', 'employmentstatus':'employment_status',
 'accounttype':'account_type', 'bankingchannel':'banking_channel', 'risktolerance':'risk_tolerance',
 'snapshotdate':'snapshot_date', 'dataorigin':'data_origin'}
NUMERIC = {'age','tenure','credit_score','balance','estimated_salary','products_number','fx_to_usd',
 'bitcoin_btc','bitcoin_price_usd','fixed_deposit','fd_rate_pct','fd_term_months','loan_balance',
 'loan_rate_pct','stocks','mutual_funds','churn','credit_card','active_member','has_bitcoin','has_fixed_deposit','has_loan'}
BINARY = {'churn','credit_card','active_member','has_bitcoin','has_fixed_deposit','has_loan'}
PROTECTED = {'customer_id','churn','surname','snapshot_date','data_origin','currency','fx_to_usd','bitcoin_price_usd'}
MONEY = {'balance','estimated_salary','fixed_deposit','loan_balance','stocks','mutual_funds'}
RANGES = {'age':(18,110), 'tenure':(0,80), 'credit_score':(300,850), 'products_number':(0,30),
 'fx_to_usd':(0.00000001,1e6), 'bitcoin_price_usd':(0.000001,1e9), 'fd_rate_pct':(0,100),
 'loan_rate_pct':(0,100), 'fd_term_months':(0,1200)}
DICTIONARY = {'customer_id':'Source customer identifier; never imputed. Repeated IDs require review.',
 'country':'Country recorded by the source; no inference from names or currencies.',
 'churn':'Observed outcome: 1 = churned, 0 = retained. Missing outcomes are excluded from rate denominators.',
 'balance':'Account balance in the row currency. Zero is a valid balance.',
 'fx_to_usd':'Snapshot exchange rate: multiply local currency by this value to obtain USD.',
 'bitcoin_btc':'Bitcoin quantity in BTC; multiply by the supplied USD price, not the FX rate.',
 'loan_balance':'Liability, shown separately from assets.',
 'estimated_salary':'Estimated salary in row currency; not bank revenue.',
 'credit_score':'Source credit score; project validation assumption: 300–850.',
 'age':'Age in years; project validation assumption: adult customers aged 18–110.',
 'tenure':'Bank relationship duration in years; must not exceed age.',
 'snapshot_date':'Snapshot date, not transaction history. Charts do not imply historical trends.',
 'data_origin':'Source provenance. The 70-country file declares synthetic classroom simulation.'}

def read_csv(source):
    df = pd.read_csv(source, dtype=str, keep_default_na=False, encoding='utf-8-sig')
    if df.empty or len(df.columns)>150 or len(df)>100000:
        raise ValueError('Use a nonempty CSV with at most 100,000 rows and 150 columns.')
    mapping = {c:ALIASES.get(re.sub(r'[^a-z0-9]', '', c.lower()), re.sub(r'\W+', '_', c.strip().lower())) for c in df}
    if len(set(mapping.values())) != len(mapping):
        raise ValueError('Multiple headers map to the same column. Rename ambiguous columns first.')
    return df.rename(columns=mapping), mapping

def blank_mask(series, column):
    s = series.astype('string').str.strip().str.lower()
    markers = MISSING | ({'none'} if column != 'loan_type' else set())
    return series.isna() | s.isin(markers)

def canonical(raw):
    """Typed analytical view. Invalid cells become NA; raw remains unchanged."""
    df = raw.copy(deep=True)
    issues = []
    for c in df:
        missing = blank_mask(df[c], c)
        s = df[c].astype('string').str.strip().mask(missing)
        invalid = pd.Series(False, index=df.index)
        if c in NUMERIC or c.endswith(('_zscore','_normalized')):
            if c in BINARY:
                s = s.str.lower().replace({'yes':'1','no':'0','true':'1','false':'0','active':'1','inactive':'0', 'churned':'1','retained':'0'})
            # Accept properly grouped thousands; never strip arbitrary punctuation.
            s = s.str.replace(r'(?<=\d),(?=\d{3}(?:\D|$))','',regex=True)
            out = pd.to_numeric(s, errors='coerce').astype(float)
            invalid = (~missing & out.isna()) | (~np.isfinite(out) & out.notna())
            if c in BINARY: invalid |= out.notna() & ~out.isin([0,1])
            if c in RANGES: invalid |= out.notna() & ~out.between(*RANGES[c])
            if c in MONEY | {'bitcoin_btc'}: invalid |= out.notna() & (out<0)
            if c in {'age','tenure','credit_score','products_number','fd_term_months'}:
                invalid |= out.notna() & (out % 1 != 0)
            df[c] = out.mask(invalid)
        else:
            if c == 'currency': s = s.str.upper()
            elif c in {'country','gender','loan_type','industry','employment_status','account_type','banking_channel','risk_tolerance'}:
                s = s.str.title().replace({'Usa':'United States','Uk':'United Kingdom','Uae':'United Arab Emirates','M':'Male','F':'Female'}) if c in {'country','gender'} else s.str.title()
            if c == 'snapshot_date':
                parsed = pd.to_datetime(s, errors='coerce', format='%Y-%m-%d')
                invalid = s.notna() & parsed.isna()
                s = s.mask(invalid)
            df[c] = s
        issues.append({'column':c,'text_or_null_missing':int(missing.sum()),'invalid_values':int(invalid.sum())})
    if {'age','tenure'} <= set(df):
        bad = df.tenure > df.age
        df.loc[bad,'tenure'] = np.nan
        for i in issues:
            if i['column']=='tenure': i['invalid_values'] += int(bad.sum())
    return df, pd.DataFrame(issues)

def profile(raw):
    df, issues = canonical(raw)
    rows=[]
    for c in df:
        s=df[c]; num=c in NUMERIC
        r={'column':c,'type':'identifier' if c=='customer_id' else 'binary' if c in BINARY else 'numeric' if num else 'text',
           'missing':int(s.isna().sum()),'missing_pct':round(100*s.isna().mean(),2), 'unique':int(s.nunique()),
           'zeros':int(s.eq(0).sum()) if num else None,'mode':None if s.mode().empty else s.mode().iloc[0]}
        if num:
            r.update({'mean':s.mean(),'median':s.median(),'std_sample':s.std(),'variance_sample':s.var(),
                'min':s.min(),'max':s.max(),'q1':s.quantile(.25),'q3':s.quantile(.75),'skew':s.skew()})
        r.update(issues.set_index('column').loc[c].to_dict()); rows.append(r)
    missing=int(df.isna().sum().sum())
    ids=df.customer_id.dropna() if 'customer_id' in df else pd.Series(dtype=str)
    return {'rows':len(df),'columns':len(df.columns),'missing_cells':missing,
      'completeness_pct':round(100*(1-missing/max(1,df.size)),2),
      'exact_duplicates':int(raw.duplicated().sum()),'normalized_duplicates':int(df.duplicated().sum()),
      'duplicate_ids':int(ids.duplicated().sum()),'profile':rows,
      'definition':'Completeness = 100 × (1 − missing or invalid cells / all cells). It is not a correctness score.'}

def recommendation(df,c):
    s=df[c].dropna()
    if c in PROTECTED or c in BINARY:
        return 'leave','Protected identity, outcome, provenance, exchange rate or binary field: keep missing and review; do not invent facts.'
    if s.empty: return 'leave','No observed values: mean, median and mode cannot be estimated.'
    if c not in NUMERIC:
        return 'unknown','Use Unknown to preserve missingness; mode is available only when majority-category substitution is justified.'
    if c in MONEY:
        if 'currency' in df:
            return 'group_median','Money is skewed and currencies differ: use the median within each currency, never one pooled local-currency mean. All-missing groups stay missing.'
        return 'median','Median resists extreme financial values; zero is valid. Imputation changes distributions and must be disclosed.'
    if c in {'bitcoin_btc','fd_rate_pct','fd_term_months','loan_rate_pct'}:
        return 'leave','Missing holdings or contract terms are not evidence of zero ownership. Retain for review.'
    skew=s.skew()
    if c in {'age','tenure','credit_score','products_number'} or pd.isna(skew) or abs(skew)>.5:
        return 'median','Median is robust to skew/outliers. For discrete fields the result is rounded to a whole value.'
    return 'mean','Mean uses all observed values and is suitable for an approximately symmetric numeric distribution. Check outliers first.'

def clean(raw, plan=None, deduplicate=True):
    df, issues=canonical(raw)
    audit=[{'timestamp':datetime.now(timezone.utc).isoformat(),'column':r['column'],'method':'normalize / validate',
       'affected':int(r['text_or_null_missing']+r['invalid_values']), 'reason':f"Text/null missing: {r['text_or_null_missing']}; invalid: {r['invalid_values']}. Invalid values quarantined as missing; raw preserved."} for r in issues.to_dict('records') if r['text_or_null_missing'] or r['invalid_values']]
    if deduplicate:
        # Remove only exact raw duplicates; never merge IDs or normalized lookalikes.
        n=int(raw.duplicated().sum()); df=df.loc[~raw.duplicated()].copy()
        audit.append({'column':'all','method':'exact duplicate removal','affected':n,'reason':'Identical source rows only; conflicting IDs retained.'})
    plan = plan if plan is not None else {c:recommendation(df,c)[0] for c in df}
    recommendations={c:recommendation(df,c) for c in df}
    for c,config in plan.items():
        if c not in df: raise ValueError(f'Unknown column: {c}')
        cfg=config if isinstance(config,dict) else {'method':config}
        method=cfg.get('method','leave'); missing=df[c].isna(); n=int(missing.sum())
        if method=='leave' or n==0: continue
        if c in PROTECTED and method!='drop': raise ValueError(f'{c} is protected. Keep missing or explicitly remove incomplete rows.')
        if method=='drop': df=df.loc[~missing].copy(); value='row removed'
        elif method in {'mean','median','group_median'}:
            if c not in NUMERIC or c in BINARY: raise ValueError('Numeric imputation requires a continuous numeric field.')
            if method=='group_median':
                if 'currency' not in df: raise ValueError('Currency column required for currency-group imputation.')
                value=df.groupby('currency',dropna=True)[c].transform('median')
            else: value=getattr(df[c],method)()
            if c in {'age','tenure','credit_score','products_number'}: value=np.round(value)
            df[c]=df[c].fillna(value)
        elif method=='mode':
            mode=df[c].mode()
            value=mode.iloc[0] if len(mode) else np.nan
            df[c]=df[c].fillna(value)
        elif method in {'unknown','constant'}:
            value='Unknown' if method=='unknown' else cfg.get('value')
            if c in NUMERIC:
                try: value=float(value)
                except (ValueError,TypeError): raise ValueError('Provide a numeric replacement.')
            df[c]=df[c].fillna(value)
        else: raise ValueError(f'Unsupported cleaning method: {method}')
        if method!='drop':
            validated,_=canonical(df)
            if (missing & df[c].notna() & validated[c].isna()).any(): raise ValueError('Replacement violates the column validation rules.')
            if c in {'age','tenure'} and {'age','tenure'}<=set(df) and (df.tenure>df.age).any():
                raise ValueError('Imputation creates tenure greater than age. Choose another method or retain missing values.')
        affected=n if method=='drop' else n-int(df[c].isna().sum())
        audit.append({'column':c,'method':method,'affected':affected,'reason':recommendations[c][1] if config==recommendations[c][0] else 'Explicit user-selected method; review assumptions before applying.',
            'replacement':'within-currency median' if method=='group_median' else str(value)})
    stamp=datetime.now(timezone.utc).isoformat()
    for a in audit: a.setdefault('timestamp',stamp)
    return df.reset_index(drop=True),audit

def transform(frame):
    df=frame.copy()
    bands={'age':([0,25,35,45,60,110],['18–25','26–35','36–45','46–60','61+'],'age_band'),
       'credit_score':([299,579,669,739,799,850],['300–579','580–669','670–739','740–799','800–850'],'credit_band'),
       'tenure':([-1,2,5,8,80],['0–2','3–5','6–8','9+'],'tenure_band')}
    for c,(bins,labels,name) in bands.items():
        if c in df: df[name]=pd.cut(df[c],bins,labels=labels).astype('string')
    for c,name,labels in [('churn','churn_label',{0:'Retained',1:'Churned'}),('active_member','activity_label',{0:'Inactive',1:'Active'}),('credit_card','card_label',{0:'No card',1:'Has card'})]:
        if c in df: df[name]=df[c].map(labels).astype('string').fillna('Unknown')
    if 'fx_to_usd' in df:
        for c in MONEY & set(df): df[c+'_usd']=df[c]*df.fx_to_usd
    if {'bitcoin_btc','bitcoin_price_usd'} <= set(df): df['bitcoin_usd']=df.bitcoin_btc*df.bitcoin_price_usd
    if 'balance_usd' in df:
        df['balance_band']=pd.cut(df.balance_usd,[-.01,0,10000,50000,100000,np.inf],labels=['Zero','0–10k','10–50k','50–100k','100k+']).astype('string')
    return df

def filter_frame(df,query=None):
    q=query or {}; masks=[]
    for c,values in q.get('categories',{}).items():
        if c in df and values:
            if pd.api.types.is_numeric_dtype(df[c]):
                masks.append(df[c].isin(pd.to_numeric(pd.Series(values),errors='coerce').dropna()))
            else: masks.append(df[c].fillna('Unknown').astype(str).isin([str(v) for v in values]))
    for c,bounds in q.get('ranges',{}).items():
        if c in df and bounds and len(bounds)==2:
            masks.append(pd.to_numeric(df[c],errors='coerce').between(float(bounds[0]),float(bounds[1])))
    if masks:
        mask=masks[0]
        for m in masks[1:]: mask=mask|m if q.get('logic')=='OR' else mask&m
        df=df.loc[mask]
    term=str(q.get('search','')).strip()
    if term: df=df.loc[df.astype('string').apply(lambda x:x.str.contains(term,case=False,regex=False,na=False)).any(axis=1)]
    sort=q.get('sort',[])
    if sort:
        cols=[s['column'] for s in sort if s['column'] in df]
        if cols: df=df.sort_values(cols,ascending=[s.get('ascending',True) for s in sort if s['column'] in df],kind='stable',na_position='last')
    return df

def segments(df,by):
    if by not in df: return pd.DataFrame()
    groups=df.groupby(by,dropna=False,observed=True)
    out=groups.size().rename('customers').to_frame()
    out['share_pct']=100*out.customers/max(1,len(df))
    if 'churn' in df:
        out['known_outcomes']=groups.churn.count(); out['churned']=groups.churn.sum(min_count=1)
        out['retained']=out.known_outcomes-out.churned
        out['churn_rate_pct']=100*out.churned/out.known_outcomes.replace(0,np.nan)
    for c in ['age','credit_score','balance_usd','estimated_salary_usd','bitcoin_usd','fixed_deposit_usd','stocks_usd','mutual_funds_usd','loan_balance_usd']:
        if c in df:
            out['mean_'+c]=groups[c].mean(); out['observed_'+c]=groups[c].count()
    out['small_sample']=out.customers<30
    return out.reset_index().sort_values('customers',ascending=False)

def outliers(df,method='IQR'):
    rows=[]
    for c in df.select_dtypes('number'):
        if c in BINARY or c=='customer_id': continue
        # Original money must be inspected within currency; compare normalized USD here.
        if c in MONEY and 'currency' in df: continue
        s=df[c]; q1=s.quantile(.25);q3=s.quantile(.75);iqr=q3-q1
        lo,hi=(q1-1.5*iqr,q3+1.5*iqr) if method=='IQR' else (s.mean()-3*s.std(),s.mean()+3*s.std())
        rows.append({'column':c,'q1':q1,'q3':q3,'iqr':iqr,'lower':lo,'upper':hi,'flagged':int(((s<lo)|(s>hi)).sum()),'method':method})
    return pd.DataFrame(rows)

def statistics(df):
    n=df.select_dtypes('number').drop(columns=['customer_id'],errors='ignore')
    if 'currency' in df: n=n.drop(columns=list(MONEY),errors='ignore')
    if n.empty: return pd.DataFrame()
    result=n.describe(percentiles=[.05,.25,.5,.75,.95]).T
    result['median']=n.median();result['variance_sample']=n.var(); result['skew']=n.skew()
    result['iqr']=n.quantile(.75)-n.quantile(.25)
    result['range']=n.max()-n.min()
    result['mode']=n.apply(lambda s: s.mode().iloc[0] if len(s.mode()) else np.nan)
    return result.reset_index(names='column')

def records(df):
    import json
    return json.loads(df.to_json(orient='records',date_format='iso'))

def excel_safe(df):
    """Prevent exported text from being interpreted as spreadsheet formulas."""
    d=df.copy()
    for c in d.select_dtypes(include=['object','string']).columns:
        d[c]=d[c].map(lambda x:"'"+x if isinstance(x,str) and x.lstrip().startswith(('=','+','-','@','\t','\r')) else x)
    return d

def workbook(df,audit,quality):
    b=io.BytesIO()
    with pd.ExcelWriter(b,engine='openpyxl') as w:
        excel_safe(df).to_excel(w,sheet_name='Active data',index=False)
        statistics(df).to_excel(w,sheet_name='Statistics',index=False)
        excel_safe(pd.DataFrame(audit)).to_excel(w,sheet_name='Cleaning audit',index=False)
        pd.DataFrame(quality['profile']).to_excel(w,sheet_name='Quality',index=False)
        if 'country' in df: excel_safe(segments(df,'country')).to_excel(w,sheet_name='Countries',index=False)
        if {'country','churn_label'}<=set(df):
            excel_safe(pd.crosstab(df.country.fillna('Unknown'),df.churn_label.fillna('Unknown')).reset_index()).to_excel(w,sheet_name='Country outcome cross-tab',index=False)
        if {'country','products_number'}<=set(df):
            excel_safe(df.groupby(['country','products_number'],dropna=False).size().reset_index(name='records')).to_excel(w,sheet_name='Product distribution',index=False)
    return b.getvalue()


## 2. Extract and inspect the actual schema
Read as strings first so pandas does not silently interpret legitimate category labels as missing. Header mapping is explicit and collisions are rejected.

In [ ]:
raw, mapping = read_csv(DATA_PATH)
raw_snapshot = raw.copy(deep=True)
display(pd.DataFrame(mapping.items(), columns=['source_header','analytical_header']))
print('Shape:', raw.shape)
display(raw.head(10))
display(raw.tail(10))
display(pd.DataFrame([{'column':c,'definition':DICTIONARY.get(c,c.replace('_',' '))} for c in raw]))
print('Provenance:', raw.get('data_origin',pd.Series(dtype=str)).unique())

## 3. Profile before choosing a cleaning method
Count textual missingness separately from invalid numeric values. Exact duplicate rows and repeated customer IDs represent different problems. Never automatically merge conflicting IDs.

In [ ]:
typed, conversion_issues = canonical(raw)
before = profile(raw)
display(pd.DataFrame(before['profile']))
display(conversion_issues)
print({k:v for k,v in before.items() if k not in ['profile','definition']})
print(before['definition'])
duplicate_ids = typed.loc[typed.customer_id.notna() & typed.customer_id.duplicated(keep=False)]
display(duplicate_ids.head(20))
invalid_examples=[]
for c in raw:
    mask = ~blank_mask(raw[c],c) & typed[c].isna()
    for row in raw.index[mask][:8]:
        invalid_examples.append({'source_row':int(row)+2,'column':c,'original_value':raw.loc[row,c]})
display(pd.DataFrame(invalid_examples))

In [ ]:
quality = pd.DataFrame(before['profile'])
px.bar(quality,x='column',y='missing_pct',title='Missing / invalid cells before cleaning (%)',labels={'missing_pct':'Missing or invalid (%)'}).show()
px.imshow(typed.isna().head(300).astype(int).T,aspect='auto',color_continuous_scale=['#e9edf5','#ee7298'],title='Missingness matrix — first 300 source rows').show()

## 4. Why mean, median or mode? A defensible decision

- **Mean**: uses every observed value; useful for approximately symmetric continuous measurements without influential extremes. It is pulled toward large outliers. Do not average identifiers, binary labels or mixed local currencies.
- **Median**: the middle observed value; more robust to skew and extreme balances. For money, calculate it **within currency**. Rounded medians preserve integer counts. It reduces variance and is still only an assumption.
- **Mode**: most frequent observed value; applies to categories or discrete values. It can inflate the largest group and conceal missingness. Ties here use pandas' first sorted mode and are not evidence of a uniquely correct replacement.
- **Unknown / keep missing**: preserves uncertainty. Default text treatment is `Unknown`. IDs, churn, binary ownership/status, currency, FX and pricing fields remain missing for review. Missing Bitcoin quantities and contract terms remain missing.
- **Remove rows**: explicit option when essential fields are absent; disclose sample loss and potential selection bias. Never silently remove valid extremes.

The plan below is computed from the observed columns and their distributions. You can edit the plan before the next cell. Exploratory imputation does not justify using the same procedure for a predictive model: fit imputers on training data only to avoid leakage.

In [ ]:
decision_rows=[]
for c in typed:
    method, reason = recommendation(typed,c)
    s=typed[c]
    decision_rows.append({'column':c,'missing':int(s.isna().sum()),'method':method,'why':reason,
        'skew':s.skew() if c in NUMERIC else None})
decisions=pd.DataFrame(decision_rows)
display(decisions)
plan = dict(zip(decisions.column, decisions.method))
# Examples of explicit overrides:
# plan['age'] = 'median'
# plan['gender'] = 'mode'  # Only if you can justify majority-category substitution.
# plan['industry'] = {'method':'constant','value':'Unknown'}
# plan['churn'] = 'drop'  # Explicitly drop unknown outcomes; never invent labels.
display(pd.DataFrame({'illustrative_values':[10,11,12,13,1000]}).agg(['mean','median']))
print('The small example above is illustrative, not banking records.')

## 5. Apply the reviewed plan to a copy
Normalize labels, quarantine invalid entries as missing, remove exact source duplicates, then apply the documented per-column strategy. Conflicting IDs remain flagged. No default outlier removal.

In [ ]:
cleaned, audit = clean(raw, plan=plan, deduplicate=True)
after = profile(cleaned)
comparison=pd.DataFrame([
    {'stage':'Before','rows':before['rows'],'missing_or_invalid_cells':before['missing_cells'],'exact_duplicates':before['exact_duplicates'],'extra_repeated_ids':before['duplicate_ids']},
    {'stage':'After','rows':after['rows'],'missing_or_invalid_cells':after['missing_cells'],'exact_duplicates':after['exact_duplicates'],'extra_repeated_ids':after['duplicate_ids']}])
display(comparison)
display(pd.DataFrame(audit))
display(pd.DataFrame(after['profile']).query('missing > 0')[['column','missing','missing_pct']])
print('Unresolved missing values are intentional when a defensible replacement is unavailable.')
pd.testing.assert_frame_equal(raw,raw_snapshot)
assert len(cleaned)==len(raw)-int(raw.duplicated().sum())
assert cleaned.churn.dropna().isin([0,1]).all()

## 6. Missing text, currency and binary cross-checks
`Unknown` is an explicit category and must not be mistaken for observed demographic information. Remaining missing outcomes are excluded from churn denominators. Missing currency is kept for review even when FX is recorded; USD conversion uses the supplied FX only.

The audit reports affected cell counts. The cell-level provenance export records raw values, typed values, final values and whether a value was replaced. Removed exact duplicates have no final row.

In [ ]:
kept = raw.loc[~raw.duplicated()].copy()
typed_kept = canonical(kept)[0].reset_index(drop=True)
provenance=[]
for c in raw:
    original = kept[c].reset_index(drop=True)
    old = typed_kept[c]
    new = cleaned[c]
    changed = old.astype('string').fillna('<MISSING>').ne(new.astype('string').fillna('<MISSING>'))
    review = changed | old.isna()
    for i in np.flatnonzero(review.to_numpy()):
        provenance.append({'source_csv_row':int(kept.index[i])+2,'customer_id':str(original.iloc[i]) if c=='customer_id' else str(cleaned.customer_id.iloc[i]),
            'column':c,'raw_value':original.iloc[i],'typed_value':old.iloc[i],'cleaned_value':new.iloc[i],
            'imputed':bool(old.isna().iloc[i] and pd.notna(new.iloc[i]))})
cell_audit=pd.DataFrame(provenance)
display(cell_audit.head(30))
print('Missing churn excluded:',int(cleaned.churn.isna().sum()))
print('Valid zero balances retained:',int(cleaned.balance.eq(0).sum()))

## 7. Transform without destroying the original scale
Derive age, tenure and credit bands. Convert local money with the row-level FX rate. Bitcoin uses BTC × USD price; it is never multiplied by the local FX rate. Loans are liabilities, excluded from assets. No invented transactions, historical returns, or current market prices.

In [ ]:
data=transform(cleaned)
display(data.head(10))
print('Derived columns:',[c for c in data if c not in cleaned])
display(outliers(data,'IQR'))
display(outliers(data,'Z-score'))
print('Outliers are retained. IQR flags can include legitimate zeros or large holdings.')
numeric=statistics(data)
display(numeric)

## 8. Segregation, dynamic sorting, filtering and extraction
Edit the controls dictionary and rerun this cell. The website provides these controls without editing code. Filters can combine with AND/OR; text search is literal, and sorting is stable. All summaries below use this same selected dataframe.

In [ ]:
controls = {
    'categories': {'country':['India','United States','Germany']},
    'ranges': {'age':[25,65]},
    'logic':'AND',
    'search':'',
    'sort':[{'column':'balance_usd','ascending':False}]
}
selected=filter_frame(data,controls)
print(f'{len(selected):,} matching records out of {len(data):,}')
display(selected.head(20))
known=selected.churn.count()
print('Known outcomes:',known,'| churn rate:', f'{100*selected.churn.mean():.2f}%' if known else 'Unavailable')
country_summary=segments(selected,'country')
display(country_summary)
for group in ['gender','age_band','credit_band','tenure_band','industry','activity_label']:
    display(segments(selected,group))
display(pd.crosstab(selected.country,selected.churn_label,dropna=False))
high_balance=selected.loc[selected.balance_usd>=100000]
inactive_churned=selected.loc[(selected.active_member==0)&(selected.churn==1)]
print('Selected high-balance records:',len(high_balance),'| inactive and churned:',len(inactive_churned))

## 9. Interactive visualization gallery
All charts in this section use `selected`, not the full source. Hover, zoom and export with the Plotly toolbar. Sample sizes and missingness matter when comparing countries. Holding more of an asset does not establish that the asset performs better.

In [ ]:
if not selected.empty:
    px.bar(country_summary,x='country',y='customers',title='Selected records by country').show()
    px.pie(selected,names='churn_label',hole=.6,title='Selected churn outcomes; Unknown shown separately').show()
    px.histogram(selected,x='age',color='churn_label',title='Selected age distribution').show()
    px.scatter(selected,x='age',y='balance_usd',color='churn_label',opacity=.45,title='Age and account balance (USD)').show()
    px.box(selected,x='country',y='balance_usd',color='country',title='Balance box plots (USD)').show()
    px.violin(selected,x='country',y='balance_usd',color='country',box=True,points=False,title='Balance violin distributions (USD)').show()
    nums=selected.select_dtypes('number').drop(columns=list(MONEY)+['customer_id'],errors='ignore')
    px.imshow(nums.corr(),zmin=-1,zmax=1,color_continuous_scale='RdBu_r',title='Pairwise-complete correlation matrix').show()
    tenure=selected.groupby('tenure',as_index=False).balance_usd.mean()
    px.line(tenure,x='tenure',y='balance_usd',markers=True,title='Balance by tenure — cross-sectional, not historical').show()
    assets=['balance_usd','bitcoin_usd','fixed_deposit_usd','stocks_usd','mutual_funds_usd']
    asset_summary=selected.groupby('country')[assets].sum(min_count=1).reset_index().melt(id_vars='country',var_name='asset',value_name='usd')
    px.bar(asset_summary,x='country',y='usd',color='asset',title='Recorded asset allocation by selected country (USD)').show()
else:
    print('No matching records. Broaden controls and rerun section 8.')

## 10. Export the evidence and reproduce the website
Exports include cleaned and transformed data, the exact filtered extraction, before/after profile, column-level decisions, cell-level provenance and an Excel report. CSV text is escaped to avoid spreadsheet formula execution.

To open the full website, run `start_dashboard.bat` from the project folder, then visit http://127.0.0.1:8765. The website uses this same pipeline with cleaning preview/apply, undo/redo, uploads, 23 chart choices, sorting, extraction and report exports.

In [ ]:
excel_safe(cleaned).to_csv(OUT/'banking_cleaned.csv',index=False)
excel_safe(data).to_csv(OUT/'banking_transformed.csv',index=False)
excel_safe(selected).to_csv(OUT/'filtered_extraction.csv',index=False)
excel_safe(cell_audit).to_csv(OUT/'cell_level_audit.csv',index=False)
decisions.to_csv(OUT/'cleaning_decisions.csv',index=False)
comparison.to_csv(OUT/'quality_before_after.csv',index=False)
pd.DataFrame(audit).to_csv(OUT/'cleaning_audit.csv',index=False)
(OUT/'analytics_report.xlsx').write_bytes(workbook(selected,audit,after))
manifest={'input_sha256':hashlib.sha256(DATA_PATH.read_bytes()).hexdigest(),'input_rows':len(raw),'output_rows':len(cleaned),
          'filters':controls,'plan':plan,'synthetic_source':True,'pandas':pd.__version__,'numpy':np.__version__}
(OUT/'run_manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
print('Saved:',*[p.name for p in OUT.iterdir() if p.is_file()],sep='\n')

## 11. Isolated missing-value demonstration
This is an explicitly artificial four-row illustration, separate from the supplied dataset. It demonstrates empty text, numeric placeholders, the preservation of zero, and currency-aware medians. It is not presented as evidence about the banking source.

In [ ]:
demo=pd.DataFrame({'customer_id':['demo1','demo2','demo3','demo4'], 'currency':['USD','USD','JPY','JPY'],
                   'balance':['0','NA','10000',' '], 'gender':['Female','  ','Male','null'], 'churn':['1','0','','1']})
demo_clean,demo_log=clean(demo,{'balance':'group_median','gender':'unknown'})
display(demo);display(demo_clean);display(pd.DataFrame(demo_log))
assert demo_clean.balance.tolist()==[0,0,10000,10000]
assert demo_clean.churn.isna().sum()==1
assert demo.loc[1,'balance']=='NA'
pd.testing.assert_frame_equal(raw,raw_snapshot)
assert segments(data,'country').customers.sum()==len(data)
assert len(selected)==len(filter_frame(data,controls))
print('Reproducibility and integrity checks passed.')

## 12. Conclusions and viva discussion
1. Report observed source defects and unresolved cells; do not claim all data is now true or complete.
2. Explain why numeric distributions, valid zeros and currency units determine cleaning choices.
3. Disclose the bias introduced by median/mode imputation; compare before/after profiles.
4. Explain repeated customer IDs separately from exact row duplicates. Dashboard counts are records, not necessarily unique people.
5. Compare asset holdings and ownership only within the selected records, with observed counts. These are not real-world investment recommendations.
6. The source is a synthetic snapshot; it cannot support real-time market or historical trend claims.
7. For production: add authenticated multi-user isolation, encrypted storage, retention controls, independent schema review, background processing and deployment monitoring. The delivered website is designed for a local academic demonstration.